# 第 59 课｜自动月报流水线与幂等交付

本 Notebook 对应「365 天量化金融」第 59 课。课程方法来自本地完整视频、课件截图和本地 Whisper 口播整理；这里把它重建为可运行、可验证的量化工程教学实现。

- 图文口播：`/Users/mojiawen/Documents/学习资料归档/_暂存/pam-video-captures/365-days-quant-finance/59_youtube-VvDcJnhVYeY/index.html`
- 环境：`pkm-quant`
- 数据：离线模拟多资产日线；生成本地 HTML 月报
- 状态：可运行学习稿；不代表生产交易系统

## 本节量化问题

月报流水线分为 Extract → Validate → Compute → Render → Publish。这里的 Publish 仅指写入本地输出目录，不向任何人或外部系统发送。

In [1]:
from pathlib import Path
import json
import os
import platform
import sys
import tempfile
import timeit

# Matplotlib 首次运行需要写字体缓存；放到可写临时目录，避免污染用户目录。
MPL_CACHE_DIR = Path(tempfile.gettempdir()) / "pkm-quant-matplotlib"
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
import seaborn as sns

# 从本机字体中选择支持中文的字体，避免标题显示成方框。
preferred_chinese_fonts = [
    "PingFang SC",
    "Arial Unicode MS",
    "Hiragino Sans GB",
    "Heiti SC",
]
available_font_names = {font.name for font in font_manager.fontManager.ttflist}
chinese_font = next(
    (name for name in preferred_chinese_fonts if name in available_font_names),
    "DejaVu Sans",
)
plt.rcParams["font.sans-serif"] = [chinese_font, "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "实验" / "p2-python-toolchain",
        cwd / "p2-python-toolchain",
        *cwd.parents,
    ]
    for candidate in candidates:
        if (candidate / "课程辅助设计.md").exists():
            return candidate
    raise FileNotFoundError("请从 p2-python-toolchain 或量化交易项目目录启动 JupyterLab")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "fixtures"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 365

print("Python:", sys.version.split()[0])
print("解释器:", sys.executable)
print("项目:", PROJECT_ROOT)
print("绘图字体:", chinese_font)

Python: 3.11.15
解释器: /opt/homebrew/Caskroom/miniforge/base/envs/pkm-quant/bin/python
项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p2-python-toolchain
绘图字体: PingFang SC


In [2]:
# Jinja2 只负责模板渲染，指标仍由独立计算函数产生。
from jinja2 import Template
import hashlib

## 1. 报告期间由 `as_of_date` 显式确定

取数据中最后一个完整月份作为报告月。`period_start/period_end` 不是“最近 30 行”，而是自然月边界。

In [3]:
# 提取最后一个完整自然月的模拟价格。
daily = pd.read_csv(DATA_DIR / "daily_prices.csv", parse_dates=["date"])
wide = daily.pivot(index="date", columns="symbol", values="adj_close").sort_index()
last_date = wide.index.max()
as_of_date = last_date
period = as_of_date.to_period("M")
period_start = period.start_time
period_end = period.end_time
monthly_prices = wide.loc[period_start:period_end]

print("报告期间:", period_start.date(), "→", period_end.date())
print("交易日:", len(monthly_prices))
assert len(monthly_prices) >= 15
assert monthly_prices.notna().all().all()

报告期间: 2025-11-01 → 2025-11-30
交易日: 17


## 2. Compute 层统一指标口径

- 月收益：`P_end / P_start - 1`；
- 日收益年化波动：`std(r_daily) × sqrt(252)`；
- 最大回撤：`min(NAV / cumulative_max(NAV) - 1)`；
- 等权基准：每日六只资产简单收益的均值。

In [4]:
# 从同一月度价格对象计算全部指标。
daily_return = monthly_prices.pct_change(fill_method=None).dropna()
nav = monthly_prices / monthly_prices.iloc[0]
drawdown = nav / nav.cummax() - 1
metrics = pd.DataFrame({
    "monthly_return": monthly_prices.iloc[-1] / monthly_prices.iloc[0] - 1,
    "annualized_volatility": daily_return.std(ddof=1) * np.sqrt(252),
    "max_drawdown": drawdown.min(),
})
benchmark_return = (1 + daily_return.mean(axis=1)).prod() - 1
metrics["excess_vs_equal_weight"] = metrics["monthly_return"] - benchmark_return
display(metrics.style.format("{:.2%}"))
assert metrics.index.equals(monthly_prices.columns)

,monthly_return,annualized_volatility,max_drawdown,excess_vs_equal_weight
symbol,,,,
BANK_A,6.82%,17.86%,-2.43%,6.64%
ENERGY_A,-5.80%,18.28%,-6.24%,-5.98%
EV_A,-3.01%,20.43%,-8.31%,-3.19%
INSURE_A,2.65%,18.23%,-3.46%,2.47%
POWER_A,0.47%,12.60%,-3.21%,0.29%
TECH_A,0.10%,21.04%,-5.07%,-0.07%


## 3. Render 层只消费已校验结果

图表、表格和文字都来自 `metrics/nav`。同一参数重复渲染应得到相同字节哈希。

In [5]:
# 保存净值图，并用 Jinja2 渲染自包含本地报告。
axis = nav.plot(figsize=(9, 4), title=f"{period} 月度净值（模拟数据）")
axis.set(xlabel="日期", ylabel="月初=1")
axis.grid(alpha=0.2)
axis.figure.tight_layout()
chart_path = OUTPUT_DIR / "59_monthly_nav.png"
axis.figure.savefig(chart_path, dpi=150)
plt.close(axis.figure)

template = Template('''<!doctype html>
<html lang="zh-CN"><meta charset="utf-8">
<title>{{ period }} 量化月报</title>
<style>body{font:16px/1.65 -apple-system,"PingFang SC",sans-serif;max-width:1000px;margin:40px auto;padding:0 24px}table{border-collapse:collapse;width:100%}th,td{padding:8px;border-bottom:1px solid #ddd;text-align:right}th:first-child,td:first-child{text-align:left}img{max-width:100%}.note{color:#666}</style>
<h1>{{ period }} 量化研究月报</h1>
<p class="note">数据：固定模拟行情；本报告不是投资建议。期间 {{ start }} 至 {{ end }}。</p>
{{ table }}
<img src="{{ chart }}" alt="月度净值图">
</html>''')

def render_report():
    return template.render(
        period=str(period), start=str(period_start.date()), end=str(period_end.date()),
        table=metrics.to_html(float_format=lambda value: f"{value:.2%}"),
        chart=chart_path.name,
    )

html_first = render_report()
html_second = render_report()
hash_first = hashlib.sha256(html_first.encode()).hexdigest()
hash_second = hashlib.sha256(html_second.encode()).hexdigest()
report_path = OUTPUT_DIR / "59_monthly_report.html"
report_path.write_text(html_first, encoding="utf-8")
print("报告:", report_path)
assert hash_first == hash_second
assert report_path.exists() and chart_path.exists()

报告: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p2-python-toolchain/outputs/59_monthly_report.html


## 4. 清单与失败闸门

空数据、未闭合期间或缺失资产应在 Validate 阶段失败，而不是发布空白报告。清单记录日期、输入哈希和产物哈希。

In [6]:
# 保存可追溯交付清单。
input_path = DATA_DIR / "daily_prices.csv"
manifest = {
    "lesson": 59,
    "as_of_date": str(as_of_date.date()),
    "period_start": str(period_start.date()),
    "period_end": str(period_end.date()),
    "input_sha256": hashlib.sha256(input_path.read_bytes()).hexdigest(),
    "report_sha256": hashlib.sha256(report_path.read_bytes()).hexdigest(),
    "report": report_path.name,
    "chart": chart_path.name,
    "status": "ready-local",
}
manifest_path = OUTPUT_DIR / "59_monthly_manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
display(pd.Series(manifest, name="value").to_frame())
assert len(manifest["report_sha256"]) == 64

,value
lesson,59
as_of_date,2025-11-25
period_start,2025-11-01
period_end,2025-11-30
input_sha256,e57dd4377afd30cec702ad33494e0a98d2a5b83c8aeccc...
report_sha256,5fd3e63a6668cd7127bab67be76e2edc92ee093044c8f5...
report,59_monthly_report.html
chart,59_monthly_nav.png
status,ready-local


## 方法边界与复用结论


- Cron 只负责触发；脚本本身仍需失败退出码、日志、告警和重跑机制。
- 自动排版成功不能证明指标正确，Compute 层应有独立测试和对账。
- 本课只生成本地文件；向外发送或发布需要单独授权。


本课首先保护数据正确性、时间一致性和可复现性，再讨论速度或自动化。局部代码通过不代表整套交易系统满足权限、监控、灾备和实盘延迟要求。